# 03c — Training Analysis

**Purpose:** Analyze training convergence and loss curves.

| Input | Output |
|---|---|
| `results/*_losses.npy` | Visualizations → `results/figures/training/` |

**Runtime:** <1 minute (CPU only, loads pre-computed losses)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

RESULTS_DIR = Path('results')
FIG_DIR = RESULTS_DIR / 'figures' / 'training'
for d in [FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 1. Load Training Losses

In [ ]:
# Try to find saved losses from main notebook
loss_files = list(RESULTS_DIR.glob('*losses*.npy'))
print(f'Found {len(loss_files)} loss files: {[f.name for f in loss_files]}')

# If not found, provide template for what to analyze
if not loss_files:
    print('\nNo loss files found. Expected files from main notebook:')
    print('  - results/ae_losses.npy (Autoencoder)')
    print('  - results/mlp_losses.npy (MLP)')
    print('  - results/vqc_losses.npy (VQC)')
    print('\nSkipping to summary (template only)')

## 2. Convergence Analysis Template

In [ ]:
# Template for plotting training curves when losses are available
# This cell will auto-populate when main notebook saves losses

import json

def plot_training_curves(losses, name, save_path):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    
    epochs = range(1, len(losses) + 1)
    
    # Left: Raw loss
    axes[0].plot(epochs, losses, 'b-', lw=1.5)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title(f'{name} Training Loss')
    axes[0].grid(True, alpha=0.3)
    
    # Right: Log scale
    axes[1].plot(epochs, losses, 'b-', lw=1.5)
    axes[1].set_yscale('log')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss (log)')
    axes[1].set_title(f'{name} Loss (log scale)')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()
    return axes

# Analysis template when files exist
if loss_files:
    for f in loss_files:
        name = f.stem.replace('_losses', '')  # 'ae', 'mlp', 'vqc'
        losses = np.load(f)
        print(f'\n=== {name.upper()} Training ===')
        print(f'  Epochs: {len(losses)}')
        print(f'  Final: {losses[-1]:.4f}')
        print(f'  Min:   {losses.min():.4f}')
        plot_training_curves_losses, name.upper(), f'{FIG_DIR}/{name}_training.pdf')

## 3. Convergence Metrics

In [ ]:
# Template metrics to compute
if loss_files:
    import json
    
    metrics = {}
    for f in loss_files:
        name = f.stem.replace('_losses', '')
        losses = np.load(f)
        
        # Convergence metrics
        final = losses[-1]
        best = losses.min()
        best_epoch = losses.argmin() + 1
        
        # Early plateau check (last 10 epochs std < 10% of range)
        plateau = np.std(losses[-10:]) < 0.1 * (losses.max() - losses.min())
        
        metrics[name] = {
            'final': float(final),
            'best': float(best),
            'best_epoch': int(best_epoch),
            'converged': bool(plateau)
        }
    
    with open(RESULTS_DIR / 'training_metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f'saved → {RESULTS_DIR}/training_metrics.json')

## 4. Summary

| Check | Status |
|:---|:---:|
| Loss files found | ✅ if files exist |
| Training curves plotted | ✅ template complete |
| Convergence metrics computed | ✅ template complete |

**Note:** This notebook requires the main notebook (`pneumonia_hybrid_qml.ipynb`) to have run and saved loss history.